## Exercise 2 HW: Introduction to PyTorch

As the first step, we need to install a suitable version of PyTorch.
Generally, it should work to simply add the `torch` package to the uv project you created in the previous week.
To have the tools to visualize results, we will also install `matplotlib`.
Open a terminal, move into the uv project directory of the previous week, then run:
```
uv add torch matplotlib
```

If you run into problems regarding the PyTorch version, go to https://pytorch.org/ and scroll down to the install section for instructions on how to install specific versions.

Alternatively, you can use [Google Colab](https://colab.research.google.com/).
This is a free service by Google that allows you to run Jupyter notebooks in their cloud.
You can get GPU access by changing the runtime type in the top bar (Runtime → Change runtime type), although a GPU is not needed for this exercise.
Their default environment has most standard packages installed, including PyTorch.

## Polynomial Regression

Polynomial regression is slightly more complex than linear regression.
Instead of modeling the outcome as a linear combination of the features, polynomial regression models it as a linear combination of powers of the features.
In the case of a single feature, i.e., inputs $x \in \mathbb{R}$, the model is defined as:
$$f(x,w) = w_0 + w_1 x + w_2 x^2 + w_3 x^3 + \dots + w_q x^q.$$

In the more general case, where $x \in \mathbb{R}^p$, a simple version of the model is defined as:
$$f(x,w) = w_0 + w_{11} x_1 + w_{12} x_1^2 + \dots + w_{1q} x_1^q + w_{21} x_2 + w_{22} x_2^2 + \dots + w_{2q} x_2^q + \dots = w_0 + \sum_{i=1}^p \sum_{j=1}^q w_{ij} x_i^j.$$
A full polynomial model of degree $q$ would additionally contain interaction terms such as $x_1 x_2$ or $x_1^2 x_3$; we omit them here.

Note that although $f$ is nonlinear in $x$, it is still *linear in the weights* $w$: polynomial regression is linear regression on the transformed features $(x, x^2, \dots, x^q)$.

In this task, you implement a one-dimensional polynomial regression model in PyTorch and study how the allowed complexity of the model (the degree $q$) affects the quality of the fit to data.

1. Import `torch` and `matplotlib.pyplot` as `plt`. The second library is commonly imported with this alias.

The following cell creates synthetic data that your model will have to fit.
The ground-truth function is a polynomial of degree 3, and the targets are corrupted with Gaussian noise (standard deviation 0.4).
There are only 10 training points and 25 validation points; all tensors have shape `(n, 1)`.

In [ ]:
torch.manual_seed(1234)
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False

n_points = 10


def make_targets(x):
    return x + 0.5 * x**2 - 0.25 * x**3 + 0.4 * torch.randn_like(x)


x = torch.linspace(-2, 3, n_points).reshape(-1, 1)
y = make_targets(x)

x_val = torch.linspace(-2, 3, 25).reshape(-1, 1) + 0.1 * torch.randn((25, 1))
y_val = make_targets(x_val)

plt.scatter(x, y, label="training")
plt.scatter(x_val, y_val, label="validation")
plt.legend()
plt.show()

2. Implement a one-dimensional polynomial regression model.
    1. Create a class `PolyRegression` that inherits from `torch.nn.Module`.
    2. Create the `__init__` method. The method should have an integer argument that specifies the degree of the polynomial that will be fitted. The weights can be created with a `torch.nn.Linear` layer, but you can also look at different approaches like using `torch.nn.parameter.Parameter`. Do not forget to call the `__init__` of the parent class first.
    3. Create the `forward` method. It receives inputs of shape `(N, 1)`, should return predictions of shape `(N, 1)`, and should implement the first equation of this notebook: $$f(x,w) = w_0 + w_1 x + w_2 x^2 + w_3 x^3 + \dots + w_q x^q.$$


Hint 1: `torch.pow` can be used to raise a tensor to a power.

Hint 2: The PyTorch documentation can help you if you are stuck.

In [ ]:
# Your implementation of PolyRegression


# Test that your model works
model = PolyRegression(3)
out = model(x)
assert out.shape == (n_points, 1), (
    f"Expected output shape {(n_points, 1)}, got {tuple(out.shape)}"
)

3. Complete the following function, which performs one training epoch. It should implement the five general training steps.

In [ ]:
def train_one_epoch(x, y, model, optimizer, loss_func):
    """Optimizes the parameters of model with one pass over the training data.

    Args:
        x (torch.Tensor): inputs
        y (torch.Tensor): targets
        model (torch.nn.Module): model to train
        optimizer (torch.optim.Optimizer): optimizer holding the model's parameters
        loss_func (callable): loss function, called as loss_func(prediction, target)

    Returns:
        float: training loss of this epoch
    """
    # Your code here


4. Complete the following function, which computes the loss on validation data. Use the `torch.no_grad` function decorator or the `torch.no_grad` context manager inside this function to avoid making computations relevant for the gradient computation.

We call the function `evaluate` rather than `eval`, since `eval` is a built-in Python function that we do not want to shadow.

In [ ]:
def evaluate(x, y, model, loss_func):
    """Computes the loss over validation data.

    Args:
        x (torch.Tensor): inputs
        y (torch.Tensor): targets
        model (torch.nn.Module): model to evaluate
        loss_func (callable): loss function, called as loss_func(prediction, target)

    Returns:
        float: validation loss
    """
    # Your code here


5. Combine the above functions to train a polynomial regression model of degree 3. The cell should instantiate a model, an optimizer, and a loss function suitable for regression, then iterate `n_epochs` times over the data to optimize the model. To ease optimization, use the `torch.optim.Adam` optimizer with `lr=5e-3`. To see whether you are overfitting, check the validation loss on `x_val, y_val` from time to time, e.g., every 500 epochs.

In [ ]:
torch.manual_seed(123)  # fixes initialization of parameters


6. Name three hyperparameters you use in your code.

*Your answer:*

7. Visualize the resulting model with the following cell. It defines a helper function `plot_model` that you can reuse in the next task.

In [ ]:
def plot_model(model, title=None):
    fig, ax = plt.subplots()
    ax.scatter(x.view(-1), y.view(-1), color="blue", label="Training data")
    ax.scatter(x_val.view(-1), y_val.view(-1), color="orange", label="Validation data")

    with torch.no_grad():
        x_plot = torch.linspace(x.min().item(), x.max().item(), 200).view(-1, 1)
        pred_plot = model(x_plot)
    ax.plot(x_plot, pred_plot, label="Model prediction", color="red")

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    if title is not None:
        ax.set_title(title)
    ax.legend()
    plt.show()


plot_model(model, title="Degree 3")

8. Revisit the model training.
    1. Train a model with a much higher degree of the polynomial, e.g., degree 9. You may need to change the number of epochs and the learning rate of the optimizer. How does the fit compare to the fit of the model with degree 3? Compare the final training *and* validation losses, not just the plots.
    2. What happens if you train a model with degree 1 or 2? Are you over- or underfitting; why?

Hint 1: Call `torch.manual_seed(123)` before creating each model so that your runs are reproducible.

Hint 2: $x$ takes values up to 3, so the feature $x^q$ quickly becomes large ($3^9 \approx 20\,000$). This makes gradient-based optimization of high-degree models slow and sensitive to the learning rate.

Hint 3: Every degree-3 polynomial is also a polynomial of higher degree (with the extra weights set to zero). After convergence, a higher-degree model must therefore reach a training loss at most as large as the degree-3 model. If your training loss is higher, the optimization has not converged yet.

In [ ]:
# 8.1: Train a model with a much higher polynomial degree
torch.manual_seed(123)


*Your answer to 8.1:*

In [ ]:
# 8.2: Train models with degree 1 and 2


*Your answer to 8.2:*

9. How would you classify this task in the scheme of slide 6 in deck 2?

*Your answer:*